In [1]:
import pandas as pd
import numpy as np

customers = pd.read_csv("E:\CODING_ALL\Data Analytics\projrcts\RetailIQ\data\Cleaned csv\cleaned_customers.csv")
sales = pd.read_csv("E:\CODING_ALL\Data Analytics\projrcts\RetailIQ\data\Cleaned csv\cleaned_sales.csv")
items = pd.read_csv("E:\CODING_ALL\Data Analytics\projrcts\RetailIQ\data\Cleaned csv\cleaned_orders.csv")

sales["order_date"] = pd.to_datetime(sales["order_date"])

# Exclude cancelled orders
eligible_sales = sales[
    sales["order_status"].str.lower() != "cancelled"
].copy()

# Attach eligible order information to order items
eligible_items = items.merge(
    eligible_sales[["order_id", "customer_id", "order_date",
                    "return_status", "delivery_days"]],
    on="order_id",
    how="inner"
)

print("Eligible orders:", eligible_sales["order_id"].nunique())
print("Eligible item rows:", len(eligible_items))

Eligible orders: 129718
Eligible item rows: 373460


In [2]:
# Reference date: one day after the latest order date
reference_date = sales["order_date"].max() + pd.Timedelta(days=1)

# Order-level revenue
order_totals = eligible_items.groupby(
    ["customer_id", "order_id", "order_date"]
).agg(
    order_value=("net_sales", "sum")
).reset_index()

# Recency and frequency
customer_features = order_totals.groupby("customer_id").agg(
    last_purchase=("order_date", "max"),
    frequency=("order_id", "nunique"),
    monetary_value=("order_value", "sum"),
    avg_order_value=("order_value", "mean")
).reset_index()

customer_features["recency_days"] = (
    reference_date - customer_features["last_purchase"]
).dt.days

# Average discount rate
discount_features = eligible_items.groupby("customer_id").agg(
    avg_discount_rate=("discount_percentage", "mean")
).reset_index()

# Average delivery time
delivery_features = eligible_sales.groupby("customer_id").agg(
    avg_delivery_days=("delivery_days", "mean")
).reset_index()

# Return rate: share of eligible orders marked as returned
eligible_sales["is_returned"] = (
    eligible_sales["return_status"]
    .fillna("")
    .str.lower()
    .isin(["returned", "refunded"])
)

return_features = eligible_sales.groupby("customer_id").agg(
    eligible_orders=("order_id", "nunique"),
    returned_orders=("is_returned", "sum")
).reset_index()

return_features["return_rate"] = (
    return_features["returned_orders"] /
    return_features["eligible_orders"]
)

# Combine all features
customer_features = (
    customer_features
    .merge(discount_features, on="customer_id", how="left")
    .merge(delivery_features, on="customer_id", how="left")
    .merge(
        return_features[["customer_id", "return_rate"]],
        on="customer_id",
        how="left"
    )
)

# Purchase frequency per day across the observed eligible period
first_purchase = order_totals.groupby("customer_id")[
    "order_date"
].min()

observation_days = (
    reference_date - customer_features["customer_id"].map(first_purchase)
).dt.days.clip(lower=1)

customer_features["purchase_frequency"] = (
    customer_features["frequency"] / observation_days
)

# Include customers with no eligible purchases
customer_features = customers[
    ["customer_id", "customer_name", "customer_segment"]
].merge(
    customer_features.drop(columns="last_purchase"),
    on="customer_id",
    how="left"
)

customer_features["frequency"] = (
    customer_features["frequency"].fillna(0).astype(int)
)
customer_features["monetary_value"] = (
    customer_features["monetary_value"].fillna(0)
)

customer_features.head()

,customer_id,customer_name,customer_segment,frequency,monetary_value,avg_order_value,recency_days,avg_discount_rate,avg_delivery_days,return_rate,purchase_frequency
0,CUST-000001,Donna Miller,Consumer,8,9469.55,1183.693750,7.0,0.121093,5.000000,0.125000,0.004477
1,CUST-000002,Joseph James,Consumer,10,18479.69,1847.969000,283.0,0.095423,5.857143,0.200000,0.005848
2,CUST-000003,Daniel Smith,Consumer,7,8268.66,1181.237143,210.0,0.086645,4.500000,0.142857,0.004152
3,CUST-000004,David Lopez,VIP,6,11092.52,1848.753333,55.0,0.192655,4.500000,0.166667,0.014599
4,CUST-000005,Dakota Davis,Consumer,8,10063.97,1257.996250,413.0,0.233239,5.125000,0.000000,0.004813


In [3]:
print("Rows:", len(customer_features))
print("Unique customers:", customer_features["customer_id"].nunique())

print(customer_features.isnull().sum())

customer_features.to_csv(
    "customer_features.csv",
    index=False
)

Rows: 25000
Unique customers: 25000
customer_id             0
customer_name           0
customer_segment        0
frequency               0
monetary_value          0
avg_order_value       124
recency_days          124
avg_discount_rate     124
avg_delivery_days     252
return_rate           124
purchase_frequency    124
dtype: int64
